# Ôn tập Buổi 02 - NumPy Foundation

        **Thời lượng gợi ý:** 60-75 phút  
        **Cách học:** trả lời câu hỏi trước khi chạy cell; sau mỗi ví dụ, tự nói thành lời *đầu vào - phép biến đổi - đầu ra*.

        ## Mục tiêu

        - Đọc shape, dtype, axis và dự đoán đầu ra của phép toán mảng.
- Dùng vectorization, broadcasting, boolean mask và fancy indexing.
- Phân biệt view/copy và giải hồi quy tuyến tính nhỏ bằng `lstsq`.

        > Notebook này là tài liệu ôn chủ động, không thay thế toàn bộ slide. Khi một câu tự kiểm tra chưa chắc, quay lại đúng mục tương ứng trong `slides/buoi2_python_datascience.pdf`.


In [1]:
from pathlib import Path

HERE = Path.cwd().resolve()
ROOT = next(
    (p for p in (HERE, *HERE.parents) if (p / "datasets").exists() and (p / "slides").exists()),
    None,
)
assert ROOT is not None, "Hãy mở notebook từ bên trong repo Hoan-Data-Science-Course."
import numpy as np
print(f"NumPy: {np.__version__}")
print(f'Repo: {ROOT}')


NumPy: 2.5.1
Repo: C:\Hon\Nam_3_HK1 2026_2027\DataScience\Hoan-Data-Science-Course


## 0. Chẩn đoán nhanh - chưa chạy code

**1. Mảng shape `(3, 4)` có `sum(axis=0)` shape gì?**

<details><summary>Kiểm tra đáp án</summary>

`(4,)`: gộp theo chiều hàng, còn lại một kết quả cho mỗi cột.

</details>

**2. Slicing NumPy thường trả về view hay copy?**

<details><summary>Kiểm tra đáp án</summary>

Thường là view; fancy indexing thường trả về copy.

</details>

**3. Hai shape `(5, 3)` và `(3,)` có broadcast được không?**

<details><summary>Kiểm tra đáp án</summary>

Có; so từ chiều cuối: `3` khớp `3`.

</details>


## 1. `ndarray`: shape, dtype, axis


In [2]:
x = np.arange(1, 13).reshape(3, 4)
print(x)
print("ndim:", x.ndim, "shape:", x.shape, "size:", x.size, "dtype:", x.dtype)
print("theo cột:", x.sum(axis=0))
print("theo hàng:", x.sum(axis=1))
assert x.sum(axis=0).shape == (4,)
assert x.sum(axis=1).shape == (3,)


[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
ndim: 2 shape: (3, 4) size: 12 dtype: int64
theo cột: [15 18 21 24]
theo hàng: [10 26 42]


## 2. Indexing, slicing và view/copy

**Đoán trước:** sửa `view[0]` có làm `a` đổi không? Còn `copied[0]`?


In [3]:
a = np.array([10, 20, 30, 40, 50])
view = a[1:4]
copied = a[[1, 2, 3]]
view[0] = 999
copied[1] = -1
print("a:", a, "| view:", view, "| copied:", copied)
assert a.tolist() == [10, 999, 30, 40, 50]
assert copied.tolist() == [20, -1, 40]


a: [ 10 999  30  40  50] | view: [999  30  40] | copied: [20 -1 40]


## 3. Vectorization, UFunc và aggregation


In [4]:
values = np.array([1.0, 4.0, 9.0, 16.0])
roots = np.sqrt(values)
standardized = (values - values.mean()) / values.std()
print(roots)
print(np.round(standardized, 3))
assert np.allclose(roots, [1, 2, 3, 4])
assert np.isclose(standardized.mean(), 0)


[1. 2. 3. 4.]
[-1.145 -0.616  0.264  1.497]


## 4. Broadcasting và boolean mask

`X - X.mean(axis=0)` trừ mean tương ứng khỏi từng cột vì vector `(3,)` được broadcast qua 4 hàng.


In [5]:
X = np.array([[10, 100, 1], [20, 120, 0], [30, 110, 1], [40, 130, 0]])
centered = X - X.mean(axis=0)
high_first_feature = X[X[:, 0] >= 30]
print(centered)
print(high_first_feature)
assert np.allclose(centered.mean(axis=0), 0)
assert high_first_feature.shape == (2, 3)


[[-15.  -15.    0.5]
 [ -5.    5.   -0.5]
 [  5.   -5.    0.5]
 [ 15.   15.   -0.5]]
[[ 30 110   1]
 [ 40 130   0]]


## 5. Sorting và vị trí


In [6]:
scores = np.array([7.5, 9.0, 6.5, 8.5])
order = np.argsort(scores)[::-1]
print("thứ tự giảm dần:", order)
print("điểm đã sắp:", scores[order])
assert order[0] == scores.argmax()


thứ tự giảm dần: [1 3 0 2]
điểm đã sắp: [9.  8.5 7.5 6.5]


## 6. Ứng dụng: hồi quy tuyến tính bằng least squares


In [7]:
x_train = np.array([0., 1., 2., 3., 4.])
y_train = np.array([1., 3., 5., 7., 9.])
design = np.column_stack([x_train, np.ones_like(x_train)])
slope, intercept = np.linalg.lstsq(design, y_train, rcond=None)[0]
predictions = design @ np.array([slope, intercept])
print(f"y = {slope:.1f}x + {intercept:.1f}")
assert np.allclose([slope, intercept], [2, 1])
assert np.allclose(predictions, y_train)


y = 2.0x + 1.0


## Bài tự luyện

        Với ma trận `M = np.arange(20).reshape(5, 4)`: chuẩn hóa từng cột về z-score, rồi lấy các hàng có giá trị cột cuối lớn hơn mean cột cuối.

        <details><summary>Gợi ý / đáp án tham khảo</summary>

        ```python
        M = np.arange(20).reshape(5, 4)
z = (M - M.mean(axis=0)) / M.std(axis=0)
selected = M[M[:, -1] > M[:, -1].mean()]
        ```

        </details>


## Phiếu rời buổi

        Không nhìn lại notebook, hãy tự xác nhận:

        - [ ] Tôi đọc đúng `axis=0` và `axis=1`.
- [ ] Tôi biết khi nào slicing có thể làm đổi mảng gốc.
- [ ] Tôi kiểm tra broadcasting bằng cách so shape từ chiều cuối.

        Nếu chưa đánh dấu được một mục, ghi lại **một ví dụ do chính bạn nghĩ ra** rồi chạy thử.
